In this notebook, we use the time varying parameters from from the `varyingerrorparams` notebook to perform the grid-scale update described in section 4.2 of the paper.

In [ ]:
#useful imports
import matplotlib.pyplot as plt
import os
import os
os.environ['PROJ_LIB'] = '/Users/rmourad/anaconda3/lib/python3.11/site-packages/pyproj/proj_dir/share/proj'
import glob
import rioxarray
import numpy as np
import pandas as pd
from tqdm import tqdm
import datetime
import warnings
from osgeo import gdal
import watertools.General.data_conversions as DC
warnings.filterwarnings('ignore')

# for the pre-trained GP model
import torch
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.variational import CholeskyVariationalDistribution
from torch.utils.data import DataLoader, TensorDataset

- load the prior params and basin-scale posteriors

In [ ]:
dfPosteriors = pd.read_csv('dfPosteriors.csv')
print(dfPosteriors)

- load the grid-scale prior precip and evaporation data

In [ ]:
emax_folder = '../data/prior/et/Emax'
emin_folder = '../data/prior/et/Emin'
emax_files = sorted(glob.glob(os.path.join(emax_folder, '*.tif')))
emin_files = sorted(glob.glob(os.path.join(emin_folder, '*.tif')))

In [ ]:
pmin_folder = '../data/prior/p/Pmin'
pmax_folder = '../data/prior/p/Pmax'
pmin_files = sorted(glob.glob(os.path.join(pmin_folder, '*.tif')))
pmax_files = sorted(glob.glob(os.path.join(pmax_folder, '*.tif')))

In [ ]:
templateE=emax_files[0]
dest_arrayE = gdal.Open(templateE)

In [ ]:
common_dates = [os.path.basename(f)[-14:-4] for f in emax_files]

In [ ]:
common_dates

### Grid-scale Posterior means

- compute the grid-scale posterior mean of $P$ and $E$

In [ ]:
for dt in tqdm(common_dates, desc="generating grid-scale posterior mean of P and E"):
    date_idx = common_dates.index(dt)
    
    #for P 
    in1 = os.path.join(pmin_folder, f'Pmin_{dt}.tif')
    in2 = os.path.join(pmax_folder, f'Pmax_{dt}.tif')
    dest_array1 = gdal.Open(in1)
    dest_array2 = gdal.Open(in2)
    Pmin = dest_array1.GetRasterBand(1).ReadAsArray()
    Pmax = dest_array2.GetRasterBand(1).ReadAsArray()
    wP = dfPosteriors.loc[date_idx, 'wP']
    mPostP = (1-wP) * Pmin + wP * Pmax
   
    
    #for E
    wE = dfPosteriors.loc[date_idx, 'wE']
    fE=0.8702
    in3 = os.path.join(emin_folder, f'Emin_{dt}.tif')
    in4 = os.path.join(emax_folder, f'Emax_{dt}.tif')
   
    dest_array3 = gdal.Open(in3)
    dest_array4 = gdal.Open(in4)
 
    Emin = dest_array3.GetRasterBand(1).ReadAsArray()
    Emax = dest_array4.GetRasterBand(1).ReadAsArray()
    mPostE = fE*((1-wE) * Emin + wE * Emax)
    
    nameP="mPostP_"+ dt+".tif"
    nameE="mPostE_"+ dt+".tif"
    outputgriddedPostP = '../data/posterior/p/mPostP'
    outputgriddedPostE = '../data/posterior/et/mPostE'
    os.makedirs(outputgriddedPostP, exist_ok=True)
    os.makedirs(outputgriddedPostE, exist_ok=True)
    outputP=os.path.join(outputgriddedPostP,nameP)
    outputE=os.path.join(outputgriddedPostE,nameE)
    geo = dest_arrayE.GetGeoTransform()
    proj = dest_arrayE.GetProjection()
    DC.Save_as_tiff(outputP, mPostP, geo, proj)
    DC.Save_as_tiff(outputE, mPostE, geo, proj)
    
    plt.figure(figsize=(6, 6))
    plt.imshow(mPostE, cmap='viridis')
    plt.colorbar()
    plt.title(f'mPostE ({dt})')
    plt.savefig(f'../figs/mPostE{dt}.png')
    plt.close()
    
    plt.figure(figsize=(6, 6))
    plt.imshow(mPostP, cmap='viridis')
    plt.colorbar()
    plt.title(f'mPostP ({dt})')
    plt.savefig(f'../figs/mPostP {dt}.png')
    plt.close()


- check if we average the grid-scale posterior mean of $P$ and $E$, we get back the basin-scale posterior mean of $P$ and $E$ from the water balance data fusion

In [ ]:
def OpenAsArrayPlot(fh, bandnumber=1, dtype='float32', nan_values=False):
    dataset = gdal.Open(fh, gdal.GA_ReadOnly)
    if dataset is None:
        raise ValueError(f"Failed to open raster file: {fh}")

    band = dataset.GetRasterBand(bandnumber)
    arr = band.ReadAsArray().astype(dtype)

    if nan_values:
        nodata_value = band.GetNoDataValue()
        if nodata_value is not None:
            arr[arr == nodata_value] = np.nan

    transform = dataset.GetGeoTransform()
    proj = dataset.GetProjection()

    dataset = None  # Close the dataset

    return arr, transform, proj

In [ ]:
# visualize one P map
mPostP_files = sorted(glob.glob(outputgriddedPostP + '/*.tif'))

mPostP_file = mPostP_files[0]  # get the path of the first file
print(mPostP_file)

Array, transform, proj = OpenAsArrayPlot(mPostP_file, nan_values=True)
# Create a figure and axis
fig, ax = plt.subplots(figsize=(4, 4))
# Display the raster using imshow
im = ax.imshow(Array, cmap='viridis', extent=[transform[0], transform[0] + transform[1] * Array.shape[1],
                                              transform[3] + transform[5] * Array.shape[0], transform[3]])
# Add a colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Precipitation (mm/month)')
# Set the labels
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
# Display the plot
plt.tight_layout()
plt.show()
mPostP_avg = np.nanmean(Array)  # get the mean values
print('avg of posterior precip mean (mm/month): {0}'.format(mPostP_avg))

In [ ]:
def OpenAsArray(fh, bandnumber=1, dtype='float32', nan_values=False):
    dataset = gdal.Open(fh, gdal.GA_ReadOnly)
    if dataset is None:
        raise ValueError(f"Failed to open raster file: {fh}")

    band = dataset.GetRasterBand(bandnumber)
    arr = band.ReadAsArray().astype(dtype)

    if nan_values:
        nodata_value = band.GetNoDataValue()
        if nodata_value is not None:
            arr[arr == nodata_value] = np.nan

    dataset = None  # Close the dataset

    return arr

In [ ]:
dates=[] 
months=[]
years=[]
mPostP_values=[]
for in_file in mPostP_files:
    filename=os.path.basename(in_file) # get file name from filehandler string
    datestr=filename[-14:-4] #get date string from file name
    print(datestr)
    dyear=int(datestr[0:4]) #get year number from date string
    print(dyear)
    dmonth=int(datestr[5:7]) #get month number from date string
    print(dmonth)
    dates.append(datetime.date(dyear,dmonth,1)) # create datetime object from year and month number
    months.append(dmonth) #append to months list
    years.append(dyear) # append to years list
    #Calculate value
    mPostP_data=OpenAsArray(in_file,nan_values=True) #open raster as numpy array    
    Average_mPostP=np.nanmean(mPostP_data) # calculate average P
    mPostP_values.append(Average_mPostP)    #append to P_values list

In [ ]:
mPostP_df=pd.DataFrame({'date':dates,'month':months,'year':years,'mPostP': mPostP_values})
mPostP_df.to_excel(r'mPostP_df.xlsx') #,sep=';'
mPostP_df

In [ ]:
mPostE_files = sorted(glob.glob(outputgriddedPostE + '/*.tif'))
dates=[] 
months=[]
years=[]
mPostE_values=[]
for in_fileE in mPostE_files:
    filename=os.path.basename(in_fileE) # get file name from filehandler string
    datestr=filename[-14:-4] #get date string from file name
    print(datestr)
    dyear=int(datestr[0:4]) #get year number from date string
    print(dyear)
    dmonth=int(datestr[5:7]) #get month number from date string
    print(dmonth)
    dates.append(datetime.date(dyear,dmonth,1)) # create datetime object from year and month number
    months.append(dmonth) #append to months list
    years.append(dyear) # append to years list
    #Calculate value
    mPostE_data=OpenAsArray(in_fileE,nan_values=True) #open raster as numpy array    
    Average_mPostE=np.nanmean(mPostE_data) # calculate average P
    mPostE_values.append(Average_mPostE)    #append to P_values list

In [ ]:
mPostE_df=pd.DataFrame({'date':dates,'month':months,'year':years,'mPostE': mPostE_values})
mPostE_df.to_excel(r'mPostE_df.xlsx') #,sep=';'
mPostE_df

### Grid-scale Posterior variances

- for evaporation

In [ ]:
def load_model(load_path, device=None):
    """Load pre-trained Gaussian Process model"""
    checkpoint = torch.load(load_path, map_location=device)
    inducing_points = checkpoint['inducing_points'].to(device)

    class SparseMaternGP(gpytorch.models.ApproximateGP):
        def __init__(self, inducing_points, nu=1.5):
            inducing_points = inducing_points.to(torch.float32)
            variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(inducing_points.size(0))
            variational_strategy = gpytorch.variational.VariationalStrategy(
                self, inducing_points, variational_distribution, learn_inducing_locations=True, jitter_val=1e-3)

            super(SparseMaternGP, self).__init__(variational_strategy)
            self.mean_module = gpytorch.means.ConstantMean()
            self.base_kernel = gpytorch.kernels.MaternKernel(nu=nu)
            self.covar_module = gpytorch.kernels.ScaleKernel(self.base_kernel)
            self.covar_module.outputscale = 1.0
            self.covar_module.requires_grad_(False)

        def forward(self, x):
            mean_x = self.mean_module(x)
            covar_x = self.covar_module(x)
            return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

    model = SparseMaternGP(inducing_points, nu=1.5).to(device)
    likelihood = GaussianLikelihood().to(device)
    
    # Load saved parameters
    model.load_state_dict(checkpoint['model_state_dict'])
    likelihood.load_state_dict(checkpoint['likelihood_state_dict'])
    model.base_kernel.lengthscale = torch.tensor(checkpoint['lengthscale']).to(device)
    model.covar_module.outputscale = torch.tensor(checkpoint['outputscale']).to(device)
    likelihood.noise = torch.tensor(checkpoint['noise']).to(device)
    model.eval()
    likelihood.eval()
    
    return model, likelihood, checkpoint['y_mean'], checkpoint['y_std']


In [ ]:
# Setup device and load model
device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
model_path = '../model/variationalgp_3ensembles_50Km.pt'
model, likelihood, y_mean, y_std = load_model(load_path=model_path, device=device)

print(f"lengthscale: {model.covar_module.base_kernel.lengthscale.item():.3f}")
print(f"outputscale: {model.covar_module.outputscale.item():.3f}")
print(f"likelihood noise: {likelihood.noise.item():.3f}")


In [ ]:
# Load template file to get coordinates
template_file = '../../generateCovarianceMatrixV/data/et/wapor/WaPOR_ETa_2003.01.01.tif'
with rioxarray.open_rasterio(template_file) as rds:
    x_coords = rds.x.values.astype(np.float32)
    y_coords = rds.y.values.astype(np.float32)
    template_data = rds.squeeze().values.astype(np.float32)

# Create spatial coordinate grid
LON, LAT = np.meshgrid(x_coords, y_coords)
spatial_coords = np.stack([LON.flatten(), LAT.flatten()], axis=1)
valid_mask = ~np.isnan(template_data.flatten())

In [14]:
def compute_SRS(model, spatial_coords, valid_mask, s_diag, batch_size=400):
    """Function to compute the covariance matrix in batches"""
    model.eval()
    valid_coords = torch.tensor(spatial_coords[valid_mask], dtype=torch.float32).to(device)
    n_valid = len(valid_coords)

    result_matrix = np.zeros((n_valid, n_valid), dtype=np.float32)
    diagonal_elements = np.zeros(n_valid, dtype=np.float32)
    
    total_sum = 0.0
    s_diag_tensor = torch.tensor(s_diag, dtype=torch.float32).to(device)
    
    with torch.no_grad():
        valid_loader = DataLoader(valid_coords, batch_size=batch_size, shuffle=False)
        
        for i, batch_x1 in enumerate(tqdm(valid_loader, desc="Computing S @ R @ S", disable=True)):
            start_i = i * batch_size
            end_i = start_i + batch_x1.size(0)
            s_batch_i = s_diag_tensor[start_i:end_i]  # diagonal elements
            
            for j, batch_x2 in enumerate(valid_loader):
                start_j = j * batch_size
                end_j = start_j + batch_x2.size(0)
                
                s_batch_j = s_diag_tensor[start_j:end_j]
                R = model.covar_module(batch_x1, batch_x2).evaluate()# Get R from the kernel
                
                # compute S @ R @ S = (s_i * R_ij * s_j)
                SR = R * s_batch_i.unsqueeze(1)  # row multiplication
                SRS_block = SR * s_batch_j.unsqueeze(0)  # column multiplications
                
                SRS_np = SRS_block.cpu().numpy()
                result_matrix[start_i:end_i, start_j:end_j] = SRS_np
                
                if i == j:
                    diag_indices = np.arange(SRS_np.shape[0])
                    diagonal_elements[start_i:end_i] = SRS_np[diag_indices, diag_indices]
                
                block_sum = np.sum(SRS_np)
                total_sum += block_sum
                
                del R, SR, SRS_block, SRS_np #clean up mem

    avg_SRS = total_sum / (n_valid * n_valid)  # avg of all elements in the cov matrix
    return result_matrix, avg_SRS

In [ ]:
dates_list = []
avg_srs_list = []

for date_str in tqdm(common_dates):
    date_idx = common_dates.index(date_str)
    emax_file = os.path.join(emax_folder, f'Emax_{date_str}.tif')
    emin_file = os.path.join(emin_folder, f'Emin_{date_str}.tif')
    
    with rioxarray.open_rasterio(emax_file) as rds:
        emax_data = rds.squeeze().values.astype(np.float32)
    with rioxarray.open_rasterio(emin_file) as rds:
        emin_data = rds.squeeze().values.astype(np.float32)
    
    emax_vals = emax_data.flatten()[valid_mask]
    emin_vals = emin_data.flatten()[valid_mask]
    
    #rE = 0.953800
    rE= dfPosteriors.loc[date_idx, 'rE']
    print("rE:",rE)

    # diagonal S (0.25 * |Emax - Emin|)
    s_diag = rE * 0.25 * np.abs(emax_vals - emin_vals) #multiplied by rE
    SRS,avg_SRS=compute_SRS(model, spatial_coords, valid_mask, s_diag, batch_size=500)
    
    dates_list.append(date_str)
    avg_srs_list.append(avg_SRS)

    vPostE = np.diag(SRS)
    
    vPostEgridscale = np.full_like(emax_data, np.nan)
    vPostEgridscaleflat = vPostEgridscale.flatten()
    validLocE = np.where(valid_mask)[0]
    vPostEgridscaleflat[validLocE] = vPostE
    vPostEgridded = vPostEgridscaleflat.reshape(emax_data.shape)
    sPostE=np.sqrt(vPostEgridded)
    
    #plot sPostE e.g.
    plt.figure(figsize=(6, 6))
    plt.imshow(sPostE, cmap='viridis')
    plt.colorbar()
    plt.title(f'sPostE_({date_str})')
    plt.savefig(f'../figs/sPostE_{date_str}.png')
    plt.close()
    
    #save gridded output
    namevE="vPostE_"+ date_str+".tif"
    namesE="sPostE_"+ date_str+".tif"
    outputgriddedvEPost = '../data/posterior/et/vPostE/'
    outputgriddedsEPost = '../data/posterior/et/sPostE/'
    os.makedirs(outputgriddedvEPost, exist_ok=True)
    os.makedirs(outputgriddedsEPost, exist_ok=True)
    
    outputvE=os.path.join(outputgriddedvEPost,namevE)
    outputsE=os.path.join(outputgriddedsEPost,namesE)
    geo = dest_arrayE.GetGeoTransform()
    proj = dest_arrayE.GetProjection()
    DC.Save_as_tiff(outputvE, vPostEgridded, geo, proj)
    DC.Save_as_tiff(outputsE, sPostE, geo, proj)
    
#check whether the avg of SRS gives us back the basin-scale variance from the wb df    
df_results = pd.DataFrame({'date': dates_list,'avg_SRS': avg_srs_list})
csv_output_path = 'vPostEgriddedavg.xlsx'
df_results.to_excel(csv_output_path, index=False)

- load precipitation gp model

In [15]:
model_path = '../model/variationalgp_Precip_3ensembles_50Km.pt'
model, likelihood, y_mean, y_std = load_model(load_path=model_path, device=device)

print(f"lengthscale: {model.covar_module.base_kernel.lengthscale.item():.3f}")
print(f"outputscale: {model.covar_module.outputscale.item():.3f}")
print(f"likelihood noise: {likelihood.noise.item():.3f}")


lengthscale: 0.450
outputscale: 1.000
likelihood noise: 0.000


In [16]:
dates_list_P = []
avg_srs_list_P = []

for date_str in tqdm(common_dates):
    date_idx = common_dates.index(date_str)
    pmax_file = os.path.join(pmax_folder, f'Pmax_{date_str}.tif')
    pmin_file = os.path.join(pmin_folder, f'Pmin_{date_str}.tif')
    
    with rioxarray.open_rasterio(pmax_file) as rds:
        pmax_data = rds.squeeze().values.astype(np.float32)
    with rioxarray.open_rasterio(pmin_file) as rds:
        pmin_data = rds.squeeze().values.astype(np.float32)
    
    pmax_vals = pmax_data.flatten()[valid_mask]
    pmin_vals = pmin_data.flatten()[valid_mask]
    
    # Get rP from dfPosteriors
    rP = dfPosteriors.loc[date_idx, 'rP']
    print("rP:", rP)

    # diagonal S (0.25 * |Pmax - Pmin|) multiplied by rP
    s_diag = rP * 0.25 * np.abs(pmax_vals - pmin_vals)
    SRS, avg_SRS = compute_SRS(model, spatial_coords, valid_mask, s_diag, batch_size=500)
    
    dates_list_P.append(date_str)
    avg_srs_list_P.append(avg_SRS)

    vPostP = np.diag(SRS)
    
    # map back to grid
    vPostPgridscale = np.full_like(pmax_data, np.nan)
    vPostPgridscaleflat = vPostPgridscale.flatten()
    validLocP = np.where(valid_mask)[0]
    vPostPgridscaleflat[validLocP] = vPostP
    vPostPgridded = vPostPgridscaleflat.reshape(pmax_data.shape)
    sPostP = np.sqrt(vPostPgridded)
    
    # plot sPostP
    plt.figure(figsize=(6, 6))
    plt.imshow(sPostP, cmap='viridis')
    plt.colorbar()
    plt.title(f'sPostP_({date_str})')
    plt.savefig(f'../figs/sPostP_{date_str}.png')
    plt.close()
    
    # save gridded output
    namevP = "vPostP_" + date_str + ".tif"
    namesP = "sPostP_" + date_str + ".tif"
    outputgriddedvPPost = '../data/posterior/p/vPostP/'
    outputgriddedsPostP = '../data/posterior/p/sPostP/'
    os.makedirs(outputgriddedvPPost, exist_ok=True)
    os.makedirs(outputgriddedsPostP, exist_ok=True)
    
    outputvP = os.path.join(outputgriddedvPPost, namevP)
    outputsP = os.path.join(outputgriddedsPostP, namesP)
    geo = dest_arrayE.GetGeoTransform()
    proj = dest_arrayE.GetProjection()
    DC.Save_as_tiff(outputvP, vPostPgridded, geo, proj)
    DC.Save_as_tiff(outputsP, sPostP, geo, proj)

# check if avg of SRS gives back basin-scale variance
df_results_P = pd.DataFrame({'date': dates_list_P, 'avg_SRS': avg_srs_list_P})
csv_output_path_P = 'vPostPgriddedavg.xlsx'
df_results_P.to_excel(csv_output_path_P, index=False)

  0%|                                                   | 0/240 [00:00<?, ?it/s]

rP: 1.0469394627404776


  0%|▏                                        | 1/240 [00:38<2:33:54, 38.64s/it]

rP: 1.0466918318965823


  1%|▎                                        | 2/240 [01:23<2:47:39, 42.27s/it]

rP: 1.0465194115635914


... (471 outputs left out)

rP: 1.0470740361407425


100%|██████████████████████████████████████▊| 239/240 [4:07:43<00:56, 56.75s/it]

rP: 1.0497070970736655


100%|███████████████████████████████████████| 240/240 [4:08:41<00:00, 62.17s/it]
